
# Deep-DNABERT: Machine Learning and Deep Learning Classifiers

This notebook implements the following classifiers for DNA 6mA site prediction:

- Naive Bayes (NB)
- K-Nearest Neighbors (KNN)
- Decision Tree (DT)
- Support Vector Machine (SVM)
- Random Forest (RF)
- XGBoost
- AdaBoost
- Deep-DNABERT (Proposed DNN)

The notebook supports:
- Hybrid feature datasets
- Performance evaluation
- ROC analysis
- Confusion matrices
- Cross-validation
- Model comparison


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import random
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier
)

from xgboost import XGBClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

print("Libraries loaded successfully")


In [ ]:

SEED = 1234

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed fixed:", SEED)


In [ ]:

# Load Hybrid Dataset

train_pos = pd.read_csv("benchmark_positive_Hybrid.csv")
train_neg = pd.read_csv("benchmark_negative_Hybrid.csv")

test_pos = pd.read_csv("independent_positive_Hybrid.csv")
test_neg = pd.read_csv("independent_negative_Hybrid.csv")

train_df = pd.concat([train_pos, train_neg], axis=0)
test_df = pd.concat([test_pos, test_neg], axis=0)

train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Training samples:", train_df.shape)
print("Testing samples:", test_df.shape)


In [ ]:

# Prepare Features

drop_cols = ["Sample_ID", "Sequence", "Label"]

X_train = train_df.drop(columns=drop_cols).values
y_train = train_df["Label"].values

X_test = test_df.drop(columns=drop_cols).values
y_test = test_df["Label"].values

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Feature preprocessing completed")
print(X_train.shape)
print(X_test.shape)


In [ ]:

# Initialize Classifiers

models = {

    "NB": GaussianNB(),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "DT": DecisionTreeClassifier(
        max_depth=10,
        random_state=SEED
    ),

    "SVM": SVC(
        kernel='rbf',
        probability=True,
        random_state=SEED
    ),

    "RF": RandomForestClassifier(
        n_estimators=200,
        random_state=SEED
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        eval_metric='logloss',
        random_state=SEED
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.5,
        random_state=SEED
    )
}

print("Classifiers initialized")


In [ ]:

# Train and Evaluate Traditional Classifiers

results = []

for name, model in models.items():

    print("="*60)
    print("Training:", name)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
    else:
        y_prob = y_pred

    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results.append([
        name,
        acc,
        precision,
        recall,
        f1,
        mcc,
        auc
    ])

    print("Accuracy:", round(acc,4))
    print("Precision:", round(precision,4))
    print("Recall:", round(recall,4))
    print("F1-score:", round(f1,4))
    print("MCC:", round(mcc,4))
    print("AUC:", round(auc,4))


In [ ]:

# Deep-DNABERT Proposed DNN

dnn_model = Sequential()

dnn_model.add(Dense(
    512,
    activation='relu',
    input_shape=(X_train.shape[1],)
))
dnn_model.add(BatchNormalization())
dnn_model.add(Dropout(0.5))

dnn_model.add(Dense(256, activation='relu'))
dnn_model.add(BatchNormalization())
dnn_model.add(Dropout(0.4))

dnn_model.add(Dense(128, activation='relu'))
dnn_model.add(BatchNormalization())
dnn_model.add(Dropout(0.4))

dnn_model.add(Dense(64, activation='relu'))
dnn_model.add(Dropout(0.3))

dnn_model.add(Dense(32, activation='relu'))

dnn_model.add(Dense(1, activation='sigmoid'))

dnn_model.compile(
    optimizer=Adam(0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

dnn_model.summary()


In [ ]:

# Train Deep-DNABERT

history = dnn_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=64,
    verbose=1
)


In [ ]:

# Evaluate Deep-DNABERT

y_prob_dnn = dnn_model.predict(X_test)
y_pred_dnn = (y_prob_dnn > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred_dnn)
precision = precision_score(y_test, y_pred_dnn)
recall = recall_score(y_test, y_pred_dnn)
f1 = f1_score(y_test, y_pred_dnn)
mcc = matthews_corrcoef(y_test, y_pred_dnn)
auc = roc_auc_score(y_test, y_prob_dnn)

results.append([
    "Deep-DNABERT",
    acc,
    precision,
    recall,
    f1,
    mcc,
    auc
])

print("Deep-DNABERT Performance")
print("Accuracy:", round(acc,4))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1-score:", round(f1,4))
print("MCC:", round(mcc,4))
print("AUC:", round(auc,4))


In [ ]:

# Results Table

results_df = pd.DataFrame(
    results,
    columns=[
        "Classifier",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "MCC",
        "AUC"
    ]
)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

results_df


In [ ]:

# Accuracy Comparison

plt.figure(figsize=(12,6))

plt.bar(
    results_df["Classifier"],
    results_df["Accuracy"]
)

plt.xticks(rotation=30)

plt.xlabel("Classifier")
plt.ylabel("Accuracy")
plt.title("Classifier Accuracy Comparison")

plt.show()


In [ ]:

# ROC Curve for Deep-DNABERT

fpr, tpr, thresholds = roc_curve(y_test, y_prob_dnn)

plt.figure(figsize=(8,6))

plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.legend()

plt.show()


In [ ]:

# Confusion Matrix

cm = confusion_matrix(y_test, y_pred_dnn)

plt.figure(figsize=(6,5))

plt.imshow(cm)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i,j],
                 ha="center",
                 va="center")

plt.show()


In [ ]:

# 5-Fold Cross Validation

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=SEED
)

scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Cross Validation Scores:")
print(scores)

print("Mean Accuracy:", np.mean(scores))


In [ ]:

# Save Results

results_df.to_csv(
    "classifier_comparison_results.csv",
    index=False
)

dnn_model.save(
    "Deep_DNABERT_Proposed_Model.h5"
)

print("Results and model saved successfully")


In [ ]:

# Experimental Analysis Block 1

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 1")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 2

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 2")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 3

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 3")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 4

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 4")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 5

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 5")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 6

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 6")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 7

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 7")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 8

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 8")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 9

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 9")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 10

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 10")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 11

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 11")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 12

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 12")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 13

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 13")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 14

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 14")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 15

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 15")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 16

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 16")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 17

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 17")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 18

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 18")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 19

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 19")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 20

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 20")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 21

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 21")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 22

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 22")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 23

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 23")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 24

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 24")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)


In [ ]:

# Experimental Analysis Block 25

feature_mean = np.mean(X_train, axis=0)
feature_std = np.std(X_train, axis=0)

print("Experimental block 25")
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape:", feature_std.shape)

top_features = feature_mean[:10]

print("Top features:")
print(top_features)
